<a href="https://colab.research.google.com/github/agentesiafiap/tiao-2026/blob/main/1TIAO/Global-Solution-2/src/ml/lstm/gs2_helios.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install -q tensorflow scikit-learn matplotlib

In [ ]:
import os
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
print(f"TensorFlow: {tf.__version__}")
print(f"GPU disponível: {tf.config.list_physical_devices('GPU')}")

# ─── Caminhos ────────────────────────────────────────────────
DATA_PATH = "/content/solar_cycle_historical.csv"   # arquivo que você fez upload
MODEL_DIR = Path("/content/model")
MODEL_DIR.mkdir(exist_ok=True)

# ─── Hiperparâmetros (modo rápido, mesmos do --fast local) ───
WINDOW_SIZE = 12
HORIZON = 6
EPOCHS = 50
BATCH_SIZE = 64
RANDOM_SEED = 42

tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

TensorFlow: 2.20.0
GPU disponível: []


In [ ]:
# ─── 1. Carregar dados ───────────────────────────────────────
print("\n[1/6] Carregando dados históricos...")
df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(df["time-tag"], format="%Y-%m")
df = df.sort_values("date").reset_index(drop=True)
df["ssn"] = df["ssn"].fillna(0).clip(lower=0).astype(float)

# Usar últimos 600 meses (~50 anos = 5 ciclos solares)
df = df.tail(600).reset_index(drop=True)
print(f"  Período: {df['date'].iloc[0].strftime('%Y-%m')} → {df['date'].iloc[-1].strftime('%Y-%m')}")
print(f"  Registros: {len(df)} | SSN médio: {df['ssn'].mean():.1f} | máximo: {df['ssn'].max():.1f}")


[1/6] Carregando dados históricos...
  Período: 1976-06 → 2026-05
  Registros: 600 | SSN médio: 86.8 | máximo: 284.5


In [ ]:
# ─── 2. Normalizar ───────────────────────────────────────────
print("\n[2/6] Normalizando SSN para [0, 1]...")
scaler = MinMaxScaler(feature_range=(0, 1))
series_norm = scaler.fit_transform(df["ssn"].values.reshape(-1, 1)).flatten()


[2/6] Normalizando SSN para [0, 1]...


In [ ]:
# ─── 3. Janelas deslizantes ──────────────────────────────────
print("\n[3/6] Criando janelas deslizantes...")
X, y = [], []
for i in range(len(series_norm) - WINDOW_SIZE - HORIZON + 1):
    X.append(series_norm[i : i + WINDOW_SIZE])
    y.append(series_norm[i + WINDOW_SIZE : i + WINDOW_SIZE + HORIZON])
X = np.array(X).reshape(-1, WINDOW_SIZE, 1)
y = np.array(y)


[3/6] Criando janelas deslizantes...


In [ ]:
# ─── 4. Split temporal ───────────────────────────────────────
n = len(X)
train_end = int(n * 0.80)
val_end   = int(n * 0.90)

X_train, y_train = X[:train_end], y[:train_end]
X_val,   y_val   = X[train_end:val_end], y[train_end:val_end]
X_test,  y_test  = X[val_end:], y[val_end:]

print(f"  Treino: {len(X_train)} | Validação: {len(X_val)} | Teste: {len(X_test)}")

  Treino: 466 | Validação: 58 | Teste: 59


In [ ]:
# ─── 5. Modelo ───────────────────────────────────────────────
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(WINDOW_SIZE, 1)),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(HORIZON),
], name="helios_lstm")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"],
)
model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "helios_lstm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 12, 64)         │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 12, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 6)              │           198 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,510 (115.27 KB)

 Trainable params: 29,510 (115.27 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# ─── 6. Treino ───────────────────────────────────────────────
callbacks = [
    EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True, verbose=1),
    ModelCheckpoint(str(MODEL_DIR / "helios_lstm_best.keras"), monitor="val_loss",
                    save_best_only=True, verbose=0),
]

print(f"\n[5/6] Treinando ({EPOCHS} épocas, EarlyStopping patience=15)...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)


[5/6] Treinando (50 épocas, EarlyStopping patience=15)...
Epoch 1/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 89ms/step - loss: 0.1393 - mae: 0.2877 - val_loss: 0.0029 - val_mae: 0.0471
Epoch 2/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.0563 - mae: 0.1733 - val_loss: 0.0251 - val_mae: 0.1511
Epoch 3/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.0345 - mae: 0.1525 - val_loss: 0.0145 - val_mae: 0.1126
Epoch 4/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.0304 - mae: 0.1353 - val_loss: 0.0101 - val_mae: 0.0918
Epoch 5/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.0281 - mae: 0.1316 - val_loss: 0.0094 - val_mae: 0.0908
Epoch 6/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 0.0281 - mae: 0.1282 - val_loss: 0.0052 - val_mae: 0.0663
Epoch 7/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.0252 - mae: 0.1199 - val_loss: 0.0057 - val_mae: 0.0691
Epoch 8/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 0.0261 - mae: 0.1235 - val_loss: 0.0044 - val_mae: 0.0594
Epoch 9/50
8/8 ━━━━━━

In [ ]:
# ─── 7. Avaliação ────────────────────────────────────────────
print("\n[6/6] Avaliando no conjunto de teste...")
y_pred_norm = model.predict(X_test, verbose=0)
y_pred = scaler.inverse_transform(y_pred_norm)
y_true = scaler.inverse_transform(y_test)

mae  = mean_absolute_error(y_true.flatten(), y_pred.flatten())
rmse = np.sqrt(mean_squared_error(y_true.flatten(), y_pred.flatten()))
print(f"  MAE:  {mae:.2f} manchas solares")
print(f"  RMSE: {rmse:.2f} manchas solares")


[6/6] Avaliando no conjunto de teste...
  MAE:  66.76 manchas solares
  RMSE: 77.14 manchas solares


In [ ]:
# ─── 8. Salvar modelo ────────────────────────────────────────
model.save(str(MODEL_DIR / "helios_lstm.keras"))
print(f"  Modelo salvo em: {MODEL_DIR / 'helios_lstm.keras'}")

with open(MODEL_DIR / "scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
print(f"  Scaler salvo em: {MODEL_DIR / 'scaler.pkl'}")

meta = {
    "window_size": WINDOW_SIZE,
    "horizon": HORIZON,
    "mae_test": float(mae),
    "rmse_test": float(rmse),
    "epochs_trained": len(history.history["loss"]),
    "feature": "ssn",
    "trained_on": "Google Colab (GPU)",
    "last_n_months": 600,
}
with open(MODEL_DIR / "model_meta.json", "w") as f:
    json.dump(meta, f, indent=2)
print(f"  Metadados salvos em: {MODEL_DIR / 'model_meta.json'}")

  Modelo salvo em: /content/model/helios_lstm.keras
  Scaler salvo em: /content/model/scaler.pkl
  Metadados salvos em: /content/model/model_meta.json


In [ ]:
# ─── 9. Gráfico de treinamento ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(history.history["loss"], label="Train Loss (MSE)", color="steelblue")
ax.plot(history.history["val_loss"], label="Val Loss (MSE)", color="orange")
ax.set_title("Curvas de Treinamento — HeliOS LSTM")
ax.set_xlabel("Época"); ax.set_ylabel("MSE"); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(history.history["mae"], label="Train MAE", color="steelblue")
ax.plot(history.history["val_mae"], label="Val MAE", color="orange")
ax.set_title(f"MAE no Teste: {mae:.2f} SSN | RMSE: {rmse:.2f} SSN")
ax.set_xlabel("Época"); ax.set_ylabel("MAE"); ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/content/model/lstm_training_metrics.png", dpi=150)
plt.show()
print("\n  Gráfico salvo em: /content/model/lstm_training_metrics.png")

print("\n" + "=" * 60)
print("  TREINAMENTO CONCLUÍDO!")
print(f"  MAE: {mae:.2f} | RMSE: {rmse:.2f} | Épocas: {len(history.history['loss'])}")
print("=" * 60)
print("\n  PRÓXIMOS PASSOS:")
print("  1. Baixe os arquivos da pasta /content/model/ (painel de arquivos)")
print("  2. Coloque em: src/ml/lstm/model/")
print("  3. Copie lstm_training_metrics.png para: docs/")


  Gráfico salvo em: /content/model/lstm_training_metrics.png

  TREINAMENTO CONCLUÍDO!
  MAE: 66.76 | RMSE: 77.14 | Épocas: 16

  PRÓXIMOS PASSOS:
  1. Baixe os arquivos da pasta /content/model/ (painel de arquivos)
  2. Coloque em: src/ml/lstm/model/
  3. Copie lstm_training_metrics.png para: docs/


In [ ]:
from google.colab import files
for fname in ["helios_lstm.keras", "scaler.pkl", "model_meta.json", "lstm_training_metrics.png"]:
    files.download(f"/content/model/{fname}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>